# Otzi Reanalysis Notebook (Template)\n\nUse this notebook as an execution ledger from this point onward.\nHeavy compute stays in shell/Docker; this notebook records commands, outputs, and quick summaries.

In [ ]:
from pathlib import Path\nimport subprocess, shlex, datetime\n\nROOT = Path('/home/jsantala/src/bio-tools')\nWD = ROOT / 'mapping' / 'tst'\nRUN_TS = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')\nprint('RUN_TS:', RUN_TS)

In [ ]:
def run(cmd, cwd=WD, tee=None):\n    print('$', cmd)\n    p = subprocess.run(cmd, shell=True, cwd=str(cwd), text=True, capture_output=True)\n    out = p.stdout + p.stderr\n    if tee:\n        Path(tee).write_text(out)\n    print(out[-4000:] if len(out) > 4000 else out)\n    if p.returncode != 0:\n        raise RuntimeError(f'command failed ({p.returncode})')\n    return out

## 1) Record Inputs

In [ ]:
BAM = '/mnt/AncientDNA/Iceman-2024/iceman.oetzi.UDG_D2049_combined.mapped_rmdup.bam'\nREF = '/home/jsantala/src/bio-tools/mapping/index/hg38p14DH3630O.fa'\nprint(BAM)\nprint(REF)

## 2) Prefix Composition

In [ ]:
cmd = r"samtools view {bam} | awk '{{p=substr($1,1,1); c[p]++}} END{{print \"M\",c[\"M\"]+0; print \"F\",c[\"F\"]+0; print \"R\",c[\"R\"]+0; print \"other\", NR-(c[\"M\"]+c[\"F\"]+c[\"R\"])}}'".format(bam=BAM)\nrun(cmd)

## 3) DeDup Pre/Post Comparison (Example)

In [ ]:
PRE = 'iceman.oetzi.UDG_D2049_combined.mapped_rmdup.pair.prim.Nsort.bam'\nPOST = '/mnt/mirrored/iceman-reanalysis/dedup_out50/iceman.oetzi.UDG_D2049_combined.mapped_rmdup.pair.prim_rmdup.Nsort.bam'\ncmd = f"join -t$'\\t' -v1 <(samtools view {PRE}) <(samtools view {POST}) | cut -c1 | uniq -c"\nrun('bash -lc ' + shlex.quote(cmd))

## 4) Removed-read BAM Construction (Example)

In [ ]:
REM_SAM = '/mnt/mirrored/iceman-reanalysis/dedup_out50/iceman.oetzi.UDG_D2049_combined.mapped_rmdup.pair.prim_rmdup.Nsort.removed.sam'\nREM_BAM = '/mnt/mirrored/iceman-reanalysis/dedup_out50/iceman.oetzi.UDG_D2049_combined.mapped_rmdup.pair.prim_rmdup.Nsort.removed.bam'\nrun(f'samtools view -H {PRE} > {REM_SAM}')\ncmd = f"join -t$'\\t' -v1 <(samtools view {PRE}) <(samtools view {POST}) >> {REM_SAM}"\nrun('bash -lc ' + shlex.quote(cmd))\nrun(f'samtools view -b -o {REM_BAM} {REM_SAM}')\nrun(f'samtools flagstat {REM_BAM}')

## 5) DeepVariant Run Ledger

In [ ]:
DV_CMD = """\ndocker run --rm --user "$(id -u):$(id -g)" \\n  -v /mnt/mirrored/iceman-reanalysis/dedup_out50:/input \\n  -v /home/jsantala:/output \\n  google/deepvariant:1.10.0 \\n  /opt/deepvariant/bin/run_deepvariant \\n  --model_type=WGS \\n  --ref=/output/src/bio-tools/mapping/index/hg38p14DH3630O.fa \\n  --reads=/input/iceman.oetzi.UDG_merge_combined.mapped_rmdup.pair.prim_rmdup.sort_rmdup.coord.bam \\n  --output_vcf=/output/iceman.vcf \\n  --output_gvcf=/output/iceman.gvcf \\n  --num_shards=8 \\n  --vcf_stats_report=true \\n  --disable_small_model=false \\n  --logging_dir=/output/logs \\n  --haploid_contigs=chrX,chrY \\n  --par_regions_bed=/input/GRCh38_PAR.bed\n"""\nprint(DV_CMD)

## 6) Post-run Quick Stats

In [ ]:
run('bcftools stats /home/jsantala/iceman.vcf > /home/jsantala/iceman.vcf.stats')\nrun('tail -n 80 /home/jsantala/iceman.vcf.stats')